<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/data_prep/01_Slakh2100_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================================================================
# [Step 1] Environment Setup & Raw Data Unzipping
# ==================================================================
import os
import subprocess
from google.colab import drive

print("🚀 환경 설정을 시작합니다...")
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

RAW_DATA_DIR = "/content/slakh_raw"
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# 필수 패키지 설치
!pip install -q soundfile pyyaml tqdm

# 1. 드라이브에 있는 원본 Slakh 압축파일을 코랩 로컬로 해제 (매우 빠름)
# 주의: 본인의 드라이브 경로에 맞게 수정하세요.
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Bass_separator/raw_data/slakh2100.zip"

if os.path.exists(DRIVE_ZIP_PATH):
    print(f"📦 압축 해제 중: {DRIVE_ZIP_PATH} -> {RAW_DATA_DIR}")
    !unzip -q "{DRIVE_ZIP_PATH}" -d "{RAW_DATA_DIR}"
    print("✅ 압축 해제 완료.")
else:
    print(f"❌ 원본 압축 파일을 찾을 수 없습니다: {DRIVE_ZIP_PATH}")


In [ ]:
# ==================================================================
# [Step 2] Slakh-utils 클론 및 Redux Split 강제 적용
# ==================================================================
os.chdir("/content")
print("🛠️ 데이터 누수(Data Leakage) 방지를 위해 Redux 버전 재분할을 시작합니다...")

try:
    if not os.path.exists("slakh-utils"):
        subprocess.run(["git", "clone", "https://github.com/ethman/slakh-utils.git"], check=True)

    split_script = "slakh-utils/splits/resplit_slakh.py"
    split_json = "slakh-utils/splits/redux.json"

    # 압축 해제 시 생성된 최상위 폴더 이름 (확인 후 필요시 수정)
    SLAKH_BASE_DIR = f"{RAW_DATA_DIR}/Slakh2100"

    if os.path.exists(SLAKH_BASE_DIR):
        subprocess.run(["python", split_script, "-d", SLAKH_BASE_DIR, "-s", split_json], check=True)
        print("✅ Redux 분할 완료 (총 1710 트랙으로 정제됨).")
    else:
        print(f"⚠️ {SLAKH_BASE_DIR} 를 찾을 수 없습니다. (경로를 확인하세요)")
except Exception as e:
    print(f"❌ Redux 재분할 실패: {e}")

In [ ]:
# ==================================================================
# [Step 3] DataOps: 멀티프로세싱 기반 데이터 파싱 및 믹스다운
# ==================================================================
import os
import yaml
import shutil
import numpy as np
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed

SOURCE_DIR = f"{RAW_DATA_DIR}/Slakh2100"
OUTPUT_DIR = "/content/slakh_eval"

def process_slakh_track(track_dir: Path, output_base_dir: Path):
    meta_path = track_dir / "metadata.yaml"
    if not meta_path.exists(): return False, f"metadata.yaml 누락"

    with open(meta_path, 'r', encoding='utf-8') as f:
        metadata = yaml.safe_load(f)

    overall_gain = metadata.get('overall_gain', 1.0)

    bass_keys = []
    other_keys = []

    for stem_name, stem_info in metadata['stems'].items():
        prog = stem_info.get('program_num', 0)
        is_drum = stem_info.get('is_drum', False)
        if not is_drum and ((32 <= prog <= 39) or prog == 43):
            bass_keys.append(stem_name)
        else:
            other_keys.append(stem_name)

    if len(bass_keys) == 0: return False, "베이스 트랙 없음"
    if len(bass_keys) > 1: return False, "베이스 트랙 중복 (변인 통제 불가)"

    bass_key = bass_keys[0]
    out_track_dir = output_base_dir / track_dir.parent.name / track_dir.name
    out_track_dir.mkdir(parents=True, exist_ok=True)

    try:
        # 1. Bass 정답 변환 (Mono)
        bass_flac = track_dir / "stems" / f"{bass_key}.flac"
        bass_audio, sr = sf.read(str(bass_flac))
        if len(bass_audio.shape) > 1: bass_audio = bass_audio.mean(axis=1)

        # 2. Bassless MR 믹스다운
        mix_audio = np.zeros_like(bass_audio)
        for stem in other_keys:
            stem_path = track_dir / "stems" / f"{stem}.flac"
            if stem_path.exists():
                audio, _ = sf.read(str(stem_path))
                if len(audio.shape) > 1: audio = audio.mean(axis=1)
                min_len = min(len(mix_audio), len(audio))
                mix_audio[:min_len] += audio[:min_len]

        # Gain 적용 및 저장
        bass_audio *= overall_gain
        mix_audio *= overall_gain
        sf.write(str(out_track_dir / "bass_gt.wav"), bass_audio, sr, subtype='PCM_16')
        sf.write(str(out_track_dir / "bassless_mr.wav"), mix_audio, sr, subtype='PCM_16')
        shutil.copy2(str(track_dir / "MIDI" / f"{bass_key}.mid"), str(out_track_dir / "bass_gt.mid"))

        # 3. E2E 평가용 원본 mix 변환
        mix_flac_path = track_dir / "mix.flac"
        if mix_flac_path.exists():
            full_mix, _ = sf.read(str(mix_flac_path))
            # E2E 모델 입력용이므로 채널 형태(Stereo/Mono) 원본 유지
            sf.write(str(out_track_dir / "mix.wav"), full_mix, sr, subtype='PCM_16')

        return True, "성공"
    except Exception as e:
        return False, f"런타임 에러: {e}"

# 실행부
out_base = Path(OUTPUT_DIR)
all_tracks = []
for split in ['train', 'validation', 'test']:
    split_dir = Path(SOURCE_DIR) / split
    if split_dir.exists():
        all_tracks.extend([d for d in split_dir.iterdir() if d.is_dir() and d.name.startswith("Track")])

print(f"🔍 총 {len(all_tracks)}개의 트랙 전처리를 시작합니다 (멀티프로세싱 가동)...")

success_count = 0
fail_logs = []
num_cores = multiprocessing.cpu_count()

with ProcessPoolExecutor(max_workers=num_cores) as executor:
    futures = {executor.submit(process_slakh_track, track, out_base): track for track in all_tracks}

    for future in tqdm(as_completed(futures), total=len(futures)):
        track_path = futures[future]
        is_success, msg = future.result()
        if is_success: success_count += 1
        else: fail_logs.append(f"[{track_path.name}] {msg}")

print(f"\n🎉 전처리 완료! 총 {success_count}개의 유효 트랙이 생성되었습니다.")

In [ ]:
# ==================================================================
# [Step 4] 결과물 압축 및 구글 드라이브 백업
# ==================================================================
import shutil

DRIVE_ZIP_OUTPUT = "/content/drive/MyDrive/Bass_separator/datasets/slakh_eval_processed.zip"
LOCAL_EVAL_DIR = "/content/slakh_eval"

print(f"🗜️ 전처리된 데이터를 압축하여 드라이브로 전송합니다... (시간이 조금 걸릴 수 있습니다)")

# shutil.make_archive를 사용해 zip 파일 생성
shutil.make_archive(
    base_name=DRIVE_ZIP_OUTPUT.replace('.zip', ''), # 확장자 제외 경로
    format='zip',
    root_dir=LOCAL_EVAL_DIR
)

print(f"✅ 구글 드라이브 백업 완료! 저장 위치: {DRIVE_ZIP_OUTPUT}")
print("이제 평가 스크립트 실행 전, 이 zip 파일만 코랩으로 불러와서 풀면 즉시 평가가 가능합니다.")